# Exercise 01: Exploring and Analyzing Protein Structures in the PDB Database

## Learning Objectives

In this exercise, you will learn to:
- Query the PDB database programmatically
- Extract and analyze structural quality metrics
- **Critically evaluate** data quality and make informed decisions
- **Interpret** structural data in biological context
- Use a multiple sequence alignment to find conserved regions and relate them to structure

## Using AI Tools

You may use AI assistants (ChatGPT, Claude, etc.) for:
- Understanding syntax and library functions
- Debugging code errors
- Generating code snippets

However, **you must demonstrate**:
- Your own biological reasoning and interpretation
- Justification for decisions (not just "AI said so")
- Critical evaluation of results

**The exercises assess your understanding and judgment, not code generation.**

**How this notebook is organized:** everything below, up to the "Your Turn" section near
the end, is a **fully worked walkthrough** on one protein, 1FSZ — code and reasoning are
already filled in, there is nothing to fill in as you go. There is exactly **one** graded
exercise in this notebook: the "Your Turn" section, where you repeat this same analysis on
a protein of your own choosing. Read the walkthrough as a worked template for that.


## Introduction and Basic Skills

We'll start by learning how to query PDB and extract structural information, using one
real protein — FtsZ, PDB entry **1FSZ** — end to end. Then, in the "Your Turn" section,
you'll repeat the same analysis on a protein of your own choosing.

In [1]:
# Check if running on Google Colab
try:
    from google.colab import drive
    is_google_colab = True
except ImportError:
    is_google_colab = False

# If on Google Colab, install the package
if is_google_colab:
    %pip install numpy==2.0.2 scipy==1.16.2 pandas==2.2.2 plotly==5.24.1 biopandas==0.4.1 pypdb==2.4 tqdm==4.67.1 py3dmol==2.4.0

# NOTE: Ignore specific warning message from ipykernel=5.5.6
import warnings
import os


In [2]:
# Import libraries
import math
import requests
import json
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pypdb
from biopandas.pdb import PandasPdb
import py3Dmol
from tqdm import tqdm


# Suppress all warnings at the Python level
warnings.filterwarnings('ignore')

# Also set environment variable to suppress warnings
os.environ['PYTHONWARNINGS'] = 'ignore'

print("✓ All libraries loaded successfully")


✓ All libraries loaded successfully


/home/yescalona/Development/structural-bioinformatics-lectures/exercises/.venv/lib/python3.12/site-packages/biopandas/pdb/pandas_pdb.py:27: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  pd_version = LooseVersion(pd.__version__)


### Meet FtsZ, and our anchor structure: 1FSZ

**FtsZ** is essential for bacterial (and archaeal) cell division: it self-assembles into
a contractile ring, the Z-ring, at the future division site and constricts to split the
cell in two. It is a structural and mechanistic homolog of eukaryotic **tubulin** — both
are GTPases that polymerize into dynamic filaments — which makes FtsZ both a textbook
case of convergent-yet-homologous cytoskeletal machinery and an attractive antibacterial
drug target (block the Z-ring, block division, without touching human tubulin directly).

FtsZ is not unique to *E. coli* — it is one of the most broadly conserved cell-division
proteins across the prokaryotic tree of life, found in nearly all bacteria **and** in many
archaea. For this whole walkthrough we anchor on one specific, well-characterized
structure: **1FSZ**, FtsZ from the archaeon *Methanocaldococcus jannaschii*.

Why this one, specifically:
- At 372 residues in the crystallized construct, it is large enough to show **two
  distinct domains** clearly: an N-terminal GTPase domain (the tubulin-homolog catalytic
  core, roughly residues 23-190) and a separate C-terminal domain (roughly 190-356) — a
  good structure to practice domain-level thinking on.
- It has its own **bound GDP** resolved in the crystal (not a crystallization
  artifact) — FtsZ, like tubulin, is a GTPase, and this is its real nucleotide.
- The crystal only ever resolved residues 23-356 — **38 residues at the very ends**
  (22 at the N-terminus, 16 at the C-terminus) were never modeled at all, and there are
  **no internal gaps** in the modeled range. You'll only spot the missing ends by noticing
  that the modeled range (23-356, 334 residues) is shorter than the full 372-residue
  construct — keep that number in mind for the B-factor section below.

Everything below — the database query, the quality classification, the visualization,
the conservation analysis — is built starting **from 1FSZ itself**, not from a separate,
disconnected search.

### The PDB, and why programmatic access matters

The RCSB PDB (Research Collaboratory for Structural Bioinformatics Protein Data Bank) is
the central open-access repository for 3D structures of biological macromolecules —
currently over 200,000 entries. You can browse and search it by hand at
[rcsb.org](https://www.rcsb.org/), and that's a fine way to look at *one* structure.

It stops being a fine way to work the moment you need to compare, filter, or classify
**many** structures at once — you can't click through 250 entries one at a time and keep
the comparison consistent. That's what a **programmatic API** is for: the PDB Search API
lets you send a single JSON query over HTTP and get every matching structure back as
structured JSON, ready to filter, sort, and turn into a table. The pattern is always the
same — **submit a query → get JSON back → parse it in code** — and you'll see this exact
pattern again later in this notebook for a completely different database (a sequence
alignment service), because it's the general shape of "talking to a big database," not a
one-off trick for the PDB specifically.

The API is documented at the [PDB Search API documentation](https://search.rcsb.org/index.html#search-api)
(with [worked examples](https://search.rcsb.org/index.html#examples)). We'll query it
directly with `requests`, and also use `pypdb`, a thin Python wrapper around the same API,
for some lookups. Later we'll use **BioPandas** to load an individual structure's atomic
coordinates into a pandas DataFrame for analysis.

### 1. Starting from 1FSZ: fetch its sequence, then search for all the FtsZ structures

Rather than searching with an arbitrary keyword or an off-species identifier, we start
from **1FSZ's own sequence** and ask the PDB: "what else in here looks like this?" That
is both a literal, checkable way to "search for all the FtsZ" and a query that is
**generic** — the exact same two cells below work for any structure, which is exactly
what you'll reuse on your own protein in the Your Turn section.

In [3]:
# Fetch 1FSZ's own sequence directly from RCSB (FASTA format)
fasta_response = requests.get("https://www.rcsb.org/fasta/entry/1FSZ")
fasta_text = fasta_response.text

# A FASTA file is one header line ('>...') followed by the sequence, possibly wrapped
# over several lines -- join everything after the header into one sequence string.
fasta_lines = fasta_text.strip().splitlines()
anchor_header = fasta_lines[0]
anchor_sequence = "".join(fasta_lines[1:])

print(anchor_header)
print(f"1FSZ sequence length: {len(anchor_sequence)} residues")
print(anchor_sequence)


>1FSZ_1|Chain A|FTSZ|Methanocaldococcus jannaschii (2190)
1FSZ sequence length: 372 residues
MKFLKNVLEEGSKLEEFNELELSPEDKELLEYLQQTKAKITVVGCGGAGNNTITRLKMEGIEGAKTVAINTDAQQLIRTKADKKILIGKKLTRGLGAGGNPKIGEEAAKESAEEIKAAIQDSDMVFITCGLGGGTGTGSAPVVAEISKKIGALTVAVVTLPFVMEGKVRMKNAMEGLERLKQHTDTLVVIPNEKLFEIVPNMPLKLAFKVADEVLINAVKGLVELITKDGLINVDFADVKAVMNNGGLAMIGIGESDSEKRAKEAVSMALNSPLLDVDIDGATGALIHVMGPEDLTLEEAREVVATVSSRLDPNATIIWGATIDENLENTVRVLLVITGVQSRIEFTDTGLKRKKLELTGIPKIGSHHHHHH


In [4]:
# Build a sequence search: "find PDB entries whose sequence resembles 1FSZ's"
search_dict = {
    "query": {
        "type": "terminal",
        "service": "sequence",
        "parameters": {
            "evalue_cutoff": 1,
            "identity_cutoff": 0.3,  # at least 30% sequence identity to 1FSZ
            "target": "pdb_protein_sequence",
            "value": anchor_sequence,
        },
    },
    "return_type": "entry",
    "request_options": {
        "paginate": {"start": 0, "rows": 100},
        "sort": [{"sort_by": "score", "direction": "desc"}],
    },
}

# Send request to PDB API
response = requests.post(
    "https://search.rcsb.org/rcsbsearch/v2/query",
    json=search_dict,
)
data = response.json()

print(f"Found {data['total_count']} structures related to 1FSZ")
print(f"Retrieved {len(data['result_set'])} in this query")


Found 252 structures related to 1FSZ
Retrieved 100 in this query


In [5]:
# Extract PDB IDs from results -- 1FSZ itself should be the top hit (score 1.0),
# since it is a perfect match to its own sequence.
pdb_ids = [entry["identifier"] for entry in data["result_set"]]
print(f"Top 10 PDB IDs by similarity to 1FSZ: {pdb_ids[:10]}")
print(f"Is 1FSZ itself in the list, and is it first? {pdb_ids[0] == '1FSZ'}")


Top 10 PDB IDs by similarity to 1FSZ: ['1FSZ', '1W58', '1W59', '1W5A', '1W5B', '2VAP', '1W5E', '9V7V', '2RHH', '2RHJ']
Is 1FSZ itself in the list, and is it first? True


### 2. Extracting Structural Information

For each structure, we can extract quality metrics like resolution, R-factors,
experimental method, etc. We are doing the same as the previous step but now via the
`pypdb` library to get detailed information about each PDB structure.

In [6]:
# Example: Get detailed info for 1FSZ itself
example_pdb = "1FSZ"
info = pypdb.get_info(example_pdb)

print(f"Structure: {example_pdb}")
print(f"Title: {info['struct']['title'][:80]}...")
print(f"Method: {info['exptl'][0]['method']}")
print(f"Year: {info['rcsb_accession_info']['deposit_date'][:4]}")

# Resolution (only for X-ray/Cryo-EM)
if "refine" in info and info["refine"]:
    resolution = info["refine"][0].get("ls_d_res_high")
    if resolution:
        print(f"Resolution: {resolution} Å")


Structure: 1FSZ
Title: CRYSTAL STRUCTURE OF THE CELL-DIVISION PROTEIN FTSZ AT 2.8A RESOLUTION...
Method: X-RAY DIFFRACTION
Year: 1997
Resolution: 2.8 Å


### 3. Batch Processing with Error Handling

When processing many structures, we need robust code that handles missing data —
not every method reports the same fields (NMR structures have no resolution, for
instance).

In [7]:
def extract_structure_info(pdb_id):
    """Extract key information from a PDB entry."""
    try:
        info = pypdb.get_info(pdb_id)

        # Basic info (always present)
        result = {
            "pdb_id": pdb_id,
            "method": info["exptl"][0]["method"],
            "year": info["rcsb_accession_info"]["deposit_date"][:4],
        }

        # Resolution (may be missing for NMR)
        if "refine" in info and info["refine"]:
            result["resolution"] = info["refine"][0].get("ls_d_res_high")
            result["r_work"] = info["refine"][0].get("ls_R_factor_R_work")
            result["r_free"] = info["refine"][0].get("ls_R_factor_R_free")
        else:
            result["resolution"] = None
            result["r_work"] = None
            result["r_free"] = None

        return result

    except Exception as e:
        print(f"Error processing {pdb_id}: {e}")
        return None


# Process the first 20 structures from our 1FSZ-rooted search
structures_data = []
for pdb_id in tqdm(pdb_ids[:20], desc="Processing structures"):
    info_row = extract_structure_info(pdb_id)
    if info_row:
        structures_data.append(info_row)

# Create DataFrame
df = pd.DataFrame(structures_data)
print(f"\n✓ Successfully processed {len(df)} structures")
df.head()


Processing structures: 100%|██████████| 20/20 [00:07<00:00,  2.76it/s]


✓ Successfully processed 20 structures


,pdb_id,method,year,resolution,r_work,r_free
0,1FSZ,X-RAY DIFFRACTION,1997,2.8,0.199,0.282
1,1W58,X-RAY DIFFRACTION,2004,2.5,0.221,0.253
2,1W59,X-RAY DIFFRACTION,2004,2.7,0.216,0.296
3,1W5A,X-RAY DIFFRACTION,2004,2.4,0.218,0.264
4,1W5B,X-RAY DIFFRACTION,2004,2.2,0.209,0.259


### 4. Basic Analysis and Visualization

In [8]:
# Summary statistics
print("=== Dataset Summary ===")
print(f"Total structures: {len(df)}")
print(f"\nBy experimental method:")
print(df["method"].value_counts())

# Resolution statistics (X-ray only)
xray_df = df[df["method"] == "X-RAY DIFFRACTION"]
resolutions = xray_df["resolution"].dropna()
if len(resolutions) > 0:
    print(
        f"\nX-ray resolution range: {resolutions.min():.2f} - {resolutions.max():.2f} Å"
    )
    print(f"Mean resolution: {resolutions.mean():.2f} Å")


=== Dataset Summary ===
Total structures: 20

By experimental method:
method
X-RAY DIFFRACTION      19
ELECTRON MICROSCOPY     1
Name: count, dtype: int64

X-ray resolution range: 1.43 - 3.19 Å
Mean resolution: 2.30 Å


In [9]:
# Simple visualization: Resolution distribution
# Plot X-ray resolutions
xray_res = df[df["method"] == "X-RAY DIFFRACTION"]["resolution"].dropna()
if len(xray_res) > 0:
    # Create histogram with plotly
    fig = go.Figure()

    # Add histogram
    fig.add_trace(
        go.Histogram(
            x=xray_res,
            nbinsx=15,
            opacity=0.7,
            name="Resolution Distribution",
            marker=dict(line=dict(color="black", width=1)),
        )
    )

    # Add mean line
    mean_res = xray_res.mean()
    fig.add_vline(
        x=mean_res,
        line_dash="dash",
        line_color="red",
        line_width=2,
        annotation_text=f"Mean: {mean_res:.2f} Å",
    )

    # Update layout
    fig.update_layout(
        title="Resolution Distribution (1FSZ-related structures)",
        xaxis_title="Resolution (Å)",
        yaxis_title="Number of Structures",
        showlegend=False,
        width=800,
        height=400,
    )

    fig.show()
else:
    print("No X-ray structures with resolution data to plot")


### 5. Quick Structure Visualization with py3Dmol

In [10]:
# Visualize the best-resolution X-ray structure in our sample
best_structure = (
    df[df["method"] == "X-RAY DIFFRACTION"].nsmallest(1, "resolution").iloc[0]
)
print(
    f"Visualizing: {best_structure['pdb_id']} (Resolution: {best_structure['resolution']:.2f} Å)"
)

view = py3Dmol.view(query=f"pdb:{best_structure['pdb_id']}", width=800, height=500)
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()


Visualizing: 4M8I (Resolution: 1.43 Å)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

### 6. Structure Analysis with BioPandas

Now let's zoom in on our anchor structure, 1FSZ itself, for a deeper analysis. We'll use
BioPandas to load its atomic coordinates into a pandas DataFrame.

In [11]:
# Fetch 1FSZ using BioPandas
pdb_id = "1FSZ"
print(f"Analyzing structure: {pdb_id}")

# Fetch PDB file using BioPandas
ppdb = PandasPdb().fetch_pdb(pdb_id)

# Get the ATOM records (protein atoms)
atoms_df = ppdb.df["ATOM"]

print(f"Total atoms in structure: {len(atoms_df)}")
print(f"Unique residues: {atoms_df['residue_number'].nunique()}")

# Display first few rows to understand the data structure
print("\nFirst few rows of atomic data:")
atoms_df.head()


Analyzing structure: 1FSZ
Total atoms in structure: 2479
Unique residues: 334

First few rows of atomic data:


,record_name,atom_number,blank_1,atom_name,alt_loc,residue_name,blank_2,chain_id,residue_number,insertion,...,x_coord,y_coord,z_coord,occupancy,b_factor,blank_4,segment_id,element_symbol,charge,line_idx
0,ATOM,1,,N,,SER,,A,23,,...,18.623,-11.552,-5.579,1.0,74.80,,,N,NaN,497
1,ATOM,2,,CA,,SER,,A,23,,...,17.707,-10.891,-4.602,1.0,70.57,,,C,NaN,498
2,ATOM,3,,C,,SER,,A,23,,...,17.275,-9.526,-5.088,1.0,70.22,,,C,NaN,499
3,ATOM,4,,O,,SER,,A,23,,...,18.004,-8.861,-5.812,1.0,71.08,,,O,NaN,500
4,ATOM,5,,CB,,SER,,A,23,,...,18.399,-10.704,-3.259,1.0,71.24,,,C,NaN,501


### 7. B-factor Analysis

B-factors (temperature factors) indicate atomic mobility and flexibility in protein
structures. Let's analyze B-factor patterns to understand protein dynamics and identify
flexible regions.

In [12]:
# Get CA atoms for B-factor analysis
ca_atoms = atoms_df[atoms_df["atom_name"] == "CA"].copy()

print(f"Analyzing B-factors for {len(ca_atoms)} residues")


Analyzing B-factors for 334 residues


**How you'd reason about this before plotting (a worked example of predict-first):**

1FSZ has an N-terminal GTPase domain (roughly residues 23-190, the tubulin-homolog
catalytic core) and a C-terminal domain (roughly residues 190-356). Before looking at any
numbers, the biologically motivated prediction is: the catalytic GTPase domain does the
core, conserved, GTP-binding/hydrolysis job — that kind of functional core is usually
**more rigid** (lower B-factor) than a more peripheral domain, which has more freedom to
move without breaking the protein's central function.

This is exactly the kind of prediction you should write down **before** running the plot
below, on your own protein, in the Your Turn section — being wrong is fine, the point is
to have a falsifiable guess on record before you see the answer.

In [13]:
# Simple B-factor vs Residue Number Plot with gaps
# Create complete sequence with None for missing residues
min_res = ca_atoms["residue_number"].min()
max_res = ca_atoms["residue_number"].max()

# Create a complete range of residue numbers
all_residues = list(range(min_res, max_res + 1))

# Create mapping of residue number to B-factor
bfactor_dict = dict(zip(ca_atoms["residue_number"], ca_atoms["b_factor"]))

# Create complete lists with None for missing residues
complete_bfactors = [bfactor_dict.get(res, None) for res in all_residues]

missing_count = complete_bfactors.count(None)
print(f"Residue range: {min_res} to {max_res}")
print(f"Present residues: {len(ca_atoms)}")
print(f"Missing residues: {missing_count}")

# Plot with None values (plotly will create gaps automatically)
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=all_residues,
        y=complete_bfactors,
        mode="lines+markers",
        name="B-factor",
        line=dict(color="blue", width=2),
        marker=dict(size=4),
        connectgaps=False,  # This ensures gaps appear as breaks
    )
)

# Add mean line for reference
mean_bfactor = ca_atoms["b_factor"].mean()
fig.add_hline(
    y=mean_bfactor,
    line_dash="dash",
    line_color="red",
    annotation_text=f"Mean: {mean_bfactor:.1f}",
)

fig.update_layout(
    title="1FSZ: B-factor vs Residue Number",
    xaxis_title="Residue Number",
    yaxis_title="B-factor (Ų)",
    width=1200,
    height=500,
    showlegend=False,
)

fig.show()

# Check the prediction above directly: mean B-factor per domain
n_domain = ca_atoms[(ca_atoms["residue_number"] >= 23) & (ca_atoms["residue_number"] <= 190)]
c_domain = ca_atoms[(ca_atoms["residue_number"] >= 191) & (ca_atoms["residue_number"] <= 356)]

print(f"\nB-factor statistics:")
print(f"Mean: {mean_bfactor:.1f} Ų")
print(f"Range: {ca_atoms['b_factor'].min():.1f} - {ca_atoms['b_factor'].max():.1f} Ų")
print(f"\nN-terminal GTPase domain (23-190) mean B-factor: {n_domain['b_factor'].mean():.1f} Ų")
print(f"C-terminal domain (191-356) mean B-factor: {c_domain['b_factor'].mean():.1f} Ų")


Residue range: 23 to 356
Present residues: 334
Missing residues: 0



B-factor statistics:
Mean: 50.6 Ų
Range: 19.0 - 99.8 Ų

N-terminal GTPase domain (23-190) mean B-factor: 38.3 Ų
C-terminal domain (191-356) mean B-factor: 63.1 Ų


**What this confirms:** the GTPase domain (~38 Ų mean) is indeed noticeably more
rigid than the C-terminal domain (~63 Ų mean) — the prediction above holds. Note also
that this plot has **zero internal gaps**: the 38 unresolved residues (22 N-terminal, 16
C-terminal) don't show up as a break *inside* this range, because they're entirely outside
it (residue numbers 1-22 and 357-372 simply never appear in `ca_atoms` at all). You only
notice they're missing by checking the modeled count (334) against the full construct
(372) — which is exactly the setup for the Frontier question below.

---

## 🔭 FRONTIER → Block B (AlphaFold)

**⏭ Signpost:** You'll fully understand *why* this happens in Lecture 6, when we cover
AlphaFold's confidence metrics (pLDDT).

**The question:** 1FSZ's crystal never resolved 38 residues at its two termini (22 at
the N-terminus, 16 at the C-terminus — see above). If we ran AlphaFold on this same
sequence, do you predict AlphaFold will be **confident or unconfident** about those two
terminal regions, and why?

**💡 Hint:** A crystal fails to resolve a region for a reason — usually because that
region is genuinely mobile/disordered in solution, not because of an experimental
mistake. Would you expect a structure-prediction model to be confident about a region
with that kind of underlying behavior?

**🎁 Low-stakes:** This is a bonus/calibration item — it does not block finishing this
exercise, and being wrong here is expected and not penalized.

**✅ Minimum viable answer:** "I predict AlphaFold will be [confident / unconfident]
about the terminal regions because ___." A one-line, defended guess is a complete answer.

**A worked answer, as a model of the reasoning (not a hint you should just copy for your
own protein in the Your Turn section):** unconfident. A region a crystal fails to resolve
at all — not just "resolved with a high B-factor," but entirely absent from the electron
density — is usually intrinsically flexible or disordered in solution, and AlphaFold's
pLDDT score is specifically trained to reflect exactly that kind of structural
uncertainty. The B-factor pattern above is a hint pointing the same direction, not proof:
the C-terminal domain is already the more mobile of the two resolved domains, and the
unresolved C-terminal tail sits right past the end of that already-more-mobile domain.

---

## Final code: classifying structural quality across the 1FSZ family

A common real task: given many structures of the same protein, which one should you
use? Below is a small, **correct, final** function for that — no bugs to find, this is
what a working version of "find the best structure" looks like — followed by a worked
walkthrough applying it to the real 1FSZ-rooted dataset from above.

This is not a fill-in-the-blank exercise: everything below is filled in and explained. The
graded version of this task, on your own protein, is in the Your Turn section.

In [14]:
def find_best_structure(uniprot_id_or_sequence, use_sequence=True):
    """Find the highest-resolution experimental structure for a protein.

    Works generically for any UniProt ID (full-text search) or sequence
    (sequence search, as used throughout this notebook for 1FSZ).
    Returns None, with an explanatory message, if nothing usable is found --
    it does not crash on missing data, and it does not silently return a
    wrong answer.
    """
    if use_sequence:
        search_dict = {
            "query": {
                "type": "terminal",
                "service": "sequence",
                "parameters": {
                    "evalue_cutoff": 1,
                    "identity_cutoff": 0.3,
                    "target": "pdb_protein_sequence",
                    "value": uniprot_id_or_sequence,
                },
            },
            "return_type": "entry",
            "request_options": {"paginate": {"start": 0, "rows": 100}},
        }
    else:
        search_dict = {
            "query": {
                "type": "terminal",
                "service": "full_text",
                "parameters": {"value": uniprot_id_or_sequence},
            },
            "return_type": "entry",
            "request_options": {"paginate": {"start": 0, "rows": 100}},
        }

    response = requests.post(
        "https://search.rcsb.org/rcsbsearch/v2/query", json=search_dict
    )
    response.raise_for_status()
    data = response.json()

    candidate_ids = [entry["identifier"] for entry in data.get("result_set", [])]
    if not candidate_ids:
        print("No candidate structures found for this query.")
        return None

    best_pdb = None
    best_resolution = None  # lower Å is better -- track the minimum, not the maximum

    for candidate_id in candidate_ids:
        try:
            info = pypdb.get_info(candidate_id)
        except Exception as e:
            print(f"Skipping {candidate_id}: could not fetch info ({e})")
            continue

        # Not every method (e.g. NMR) reports a resolution -- skip those safely
        # instead of crashing on a missing key.
        if "refine" not in info or not info["refine"]:
            continue
        resolution = info["refine"][0].get("ls_d_res_high")
        if resolution is None:
            continue

        if best_resolution is None or resolution < best_resolution:
            best_resolution = resolution
            best_pdb = candidate_id

    if best_pdb is None:
        print("No candidate structure reported a usable resolution value.")
        return None

    return best_pdb, best_resolution


**Why this version works, compared to a naive first attempt:**
- It compares resolution values looking for the **minimum** — in Å, *lower* is better
  (a common first mistake is to search for the maximum instead).
- It only paginates through the JSON response it actually got (`request_options`), rather
  than assuming the first page contains everything.
- It skips entries missing a `refine`/resolution field (NMR structures, for instance)
  instead of raising a `KeyError` and crashing the whole batch.
- It wraps the per-structure `pypdb.get_info` call in its own `try/except`, so one bad
  entry doesn't stop the rest of the batch from being processed — the same pattern as
  `extract_structure_info` above.
- It supports both a UniProt-ID text search and a sequence search, so it is reusable
  as-is on your own protein in the Your Turn section, whichever kind of query you have.

In [15]:
# Apply it to our real 1FSZ-rooted dataset
best_pdb, best_res = find_best_structure(anchor_sequence, use_sequence=True)
print(f"Best-resolution structure in the 1FSZ family: {best_pdb} ({best_res:.2f} Å)")


Best-resolution structure in the 1FSZ family: 6RVQ (1.14 Å)


### A structural-quality classification, built on real data

Instead of a made-up table, let's classify the real structures already sitting in `df`
(the 20 structures batch-processed earlier) using simple, defensible resolution tiers.

In [16]:
def classify_quality(row):
    if row["method"] != "X-RAY DIFFRACTION" or pd.isna(row["resolution"]):
        return "no resolution metric (use ensemble spread / local confidence instead)"
    if row["resolution"] < 2.0:
        return "excellent (< 2.0 Å) -- good for drug design / precise side-chain geometry"
    if row["resolution"] < 3.0:
        return "good (2.0-3.0 Å) -- reliable backbone, side-chain placement less certain"
    return "moderate (> 3.0 Å) -- backbone topology only, don't trust fine detail"


df["quality_tier"] = df.apply(classify_quality, axis=1)
df[["pdb_id", "method", "resolution", "r_free", "quality_tier"]].sort_values(
    "resolution", na_position="last"
)


,pdb_id,method,resolution,r_free,quality_tier
16,4M8I,X-RAY DIFFRACTION,1.430,0.20616,excellent (< 2.0 Å) -- good for drug design / ...
5,2VAP,X-RAY DIFFRACTION,1.700,0.20900,excellent (< 2.0 Å) -- good for drug design / ...
13,2VXY,X-RAY DIFFRACTION,1.700,0.21800,excellent (< 2.0 Å) -- good for drug design / ...
19,3VOA,X-RAY DIFFRACTION,1.730,0.22990,excellent (< 2.0 Å) -- good for drug design / ...
9,2RHJ,X-RAY DIFFRACTION,1.761,0.20800,excellent (< 2.0 Å) -- good for drug design / ...
8,2RHH,X-RAY DIFFRACTION,2.001,0.25700,"good (2.0-3.0 Å) -- reliable backbone, side-ch..."
15,3WGJ,X-RAY DIFFRACTION,2.179,0.23520,"good (2.0-3.0 Å) -- reliable backbone, side-ch..."
4,1W5B,X-RAY DIFFRACTION,2.200,0.25900,"good (2.0-3.0 Å) -- reliable backbone, side-ch..."
17,3VO8,X-RAY DIFFRACTION,2.255,0.23700,"good (2.0-3.0 Å) -- reliable backbone, side-ch..."
3,1W5A,X-RAY DIFFRACTION,2.400,0.26400,"good (2.0-3.0 Å) -- reliable backbone, side-ch..."


### Worked critical evaluation: which structure for which purpose?

**Scenario:** you're planning a drug-design project on the FtsZ family, and separately
you want to study its flexibility. Which structure(s) from the table above would you
pick for each, and why?

**A worked answer (a model of the reasoning, not a template to copy verbatim for your own
protein):**
- **For drug design**, pick the structure `find_best_structure` returned above
  (`best_pdb`, at `best_res` Å) — the tightest resolution means the most precise
  side-chain and pocket geometry, which matters directly for docking or structure-based
  design. R-free (not just resolution) is worth checking too: a low resolution number
  with a high R-free means the model doesn't actually explain the diffraction data well,
  despite looking sharp on paper.
- **For studying flexibility**, resolution is a much weaker signal than the B-factor
  section above already showed — even a single well-resolved structure like 1FSZ itself
  gives you a real per-residue flexibility profile via its B-factors, and cross-checking
  against **several** structures of the same protein (do they agree on which regions are
  most mobile?) is more informative than picking the "best" single one by resolution
  alone.
- **The general lesson:** "best resolution" and "best structure for this specific
  question" are not always the same structure — the quality tiers above are a starting
  filter, not a final verdict, and the same limitations-vs-purpose reasoning applies
  whatever protein you're working with.

---

## 8. Multiple sequence alignment: where is FtsZ conserved?

So far we've asked "how good is this structure?" Now let's ask a different question:
"which *parts* of the structure does evolution care most about?" A **multiple sequence
alignment (MSA)** across several FtsZ relatives lets us find columns that stayed
identical across large evolutionary distances — a strong signal that a residue matters
functionally, independent of any single structure's quality.

This reuses the exact same idea as the PDB Search API above — **submit a query, poll or
paginate, parse the JSON/text that comes back** — just aimed at a different database: the
EBI's **Job Dispatcher**, a REST API in front of bioinformatics tools including Clustal
Omega (for alignment). We already have the raw material: `pdb_ids`, the full list of
FtsZ-family hits from the sequence search at the top of this notebook.

**Choosing which sequences to align (a rule that has to work for any protein, not just
1FSZ):** aligning all ~250 hits would be slow to run and hard to read; aligning too few
makes "conserved" nearly meaningless (with only 2-3 sequences, almost everything looks
conserved by chance). We use a simple, generic rule: take up to **10** sequences, spread
evenly across the *ranked* hit list (from the closest match to the more distant ones,
not just the top 10 nearly-identical entries) so the alignment spans a real range of
evolutionary distance. If fewer than 3 hits exist at all (possible for an obscure
protein in the Your Turn section), we skip the MSA rather than run a meaningless
alignment on 1-2 sequences.

In [17]:
# Select up to 10 sequences, evenly spread across the ranked hit list (not just the
# top 10 nearest matches), so the alignment spans real evolutionary distance.
MAX_SEQUENCES = 10
MIN_SEQUENCES = 3

if len(pdb_ids) < MIN_SEQUENCES:
    print(
        f"Only {len(pdb_ids)} related structure(s) found -- too few for a meaningful "
        f"alignment (need at least {MIN_SEQUENCES}). Skipping the MSA step."
    )
    msa_subset_ids = []
else:
    n_available = len(pdb_ids)
    n_selected = min(MAX_SEQUENCES, n_available)
    # Evenly spaced indices from 0 to n_available-1, always including the first hit
    # (1FSZ itself) and spreading the rest across the full ranked list.
    indices = sorted({round(i * (n_available - 1) / (n_selected - 1)) for i in range(n_selected)})
    msa_subset_ids = [pdb_ids[i] for i in indices]

print(f"Selected {len(msa_subset_ids)} sequences for the alignment: {msa_subset_ids}")


Selected 10 sequences for the alignment: ['1FSZ', '2RHO', '3WGN', '6RVM', '7OHN', '5H5H', '14UJ', '14UU', '14VF', '14VQ']


In [18]:
# Fetch each selected structure's sequence as FASTA and combine into one multi-FASTA
multi_fasta_parts = []
for entry_id in msa_subset_ids:
    r = requests.get(f"https://www.rcsb.org/fasta/entry/{entry_id}")
    multi_fasta_parts.append(r.text.strip())

multi_fasta_text = "\n".join(multi_fasta_parts) + "\n"
print(multi_fasta_text[:300], "...")


>1FSZ_1|Chain A|FTSZ|Methanocaldococcus jannaschii (2190)
MKFLKNVLEEGSKLEEFNELELSPEDKELLEYLQQTKAKITVVGCGGAGNNTITRLKMEGIEGAKTVAINTDAQQLIRTKADKKILIGKKLTRGLGAGGNPKIGEEAAKESAEEIKAAIQDSDMVFITCGLGGGTGTGSAPVVAEISKKIGALTVAVVTLPFVMEGKVRMKNAMEGLERLKQHTDTLVVIPNEKLFEIVPNMPLKLAFKVADEVLINAVKGLVELITKDGLINVDFADVKAV ...


In [19]:
import time

# Submit the alignment job to EBI's Clustal Omega REST API.
# NOTE: EBI's Job Dispatcher requires a contact email with every submission.
EBI_EMAIL = "your.email@example.com"  # <-- replace with your own email address

clustalo_run_url = "https://www.ebi.ac.uk/Tools/services/rest/clustalo/run"

if msa_subset_ids:
    submit_response = requests.post(
        clustalo_run_url,
        data={
            "email": EBI_EMAIL,
            "stype": "protein",
            "outfmt": "fa",  # aligned FASTA -- easiest format to parse back in Python
            "sequence": multi_fasta_text,
        },
    )
    submit_response.raise_for_status()
    job_id = submit_response.text.strip()
    print(f"Submitted Clustal Omega job: {job_id}")

    # Poll for completion -- shared EBI queue, so this can take a minute or more,
    # especially if many students submit jobs around the same time.
    status_url = f"https://www.ebi.ac.uk/Tools/services/rest/clustalo/status/{job_id}"
    status = None
    for attempt in range(30):  # up to ~5 minutes at 10s intervals
        status = requests.get(status_url).text.strip()
        print(f"  poll {attempt + 1}: {status}")
        if status in ("FINISHED", "FAILURE", "ERROR", "NOT_FOUND"):
            break
        time.sleep(10)

    if status != "FINISHED":
        print(f"Alignment did not finish successfully (status: {status}). Skipping conservation analysis.")
        alignment_fasta = None
    else:
        result_url = f"https://www.ebi.ac.uk/Tools/services/rest/clustalo/result/{job_id}/aln-fasta"
        alignment_fasta = requests.get(result_url).text
        print("Alignment retrieved.")
else:
    alignment_fasta = None


Submitted Clustal Omega job: clustalo-R20260808-095105-0537-52876443-p1m
  poll 1: RUNNING
  poll 2: RUNNING
  poll 3: RUNNING
  poll 4: RUNNING
  poll 5: RUNNING
  poll 6: RUNNING
  poll 7: RUNNING
  poll 8: RUNNING
  poll 9: RUNNING
  poll 10: RUNNING
  poll 11: RUNNING
  poll 12: RUNNING
  poll 13: RUNNING
  poll 14: RUNNING
  poll 15: RUNNING
  poll 16: RUNNING
  poll 17: RUNNING
  poll 18: RUNNING
  poll 19: RUNNING
  poll 20: RUNNING
  poll 21: RUNNING
  poll 22: RUNNING
  poll 23: RUNNING
  poll 24: RUNNING
  poll 25: RUNNING
  poll 26: RUNNING
  poll 27: RUNNING
  poll 28: RUNNING
  poll 29: RUNNING
  poll 30: RUNNING
Alignment did not finish successfully (status: RUNNING). Skipping conservation analysis.


**🔭 Optional, low-stakes:** everything above finds homologs *already in the PDB* via
a structure-sequence search. If you wanted to search much further — the wider tubulin
superfamily, or sequences that have never been crystallized at all — the tool for that is
**PSI-BLAST** (also available as an EBI REST service), which iteratively searches large
sequence databases like UniProt/NR rather than just the PDB. It is not used here: it is
slower and its runtime is far less predictable than the structure-sequence search above,
which makes it a poor fit for a live class exercise. Worth knowing it exists for your own
later work, not required here.

In [20]:
# Parse the aligned FASTA and compute per-column conservation, anchored on 1FSZ
def parse_fasta_alignment(fasta_text):
    records = {}
    current_id = None
    current_seq = []
    for line in fasta_text.strip().splitlines():
        if line.startswith(">"):
            if current_id is not None:
                records[current_id] = "".join(current_seq)
            current_id = line[1:].split("|")[0]
            current_seq = []
        else:
            current_seq.append(line)
    if current_id is not None:
        records[current_id] = "".join(current_seq)
    return records


conserved_residues = []  # list of (1FSZ_residue_number, amino_acid, conservation_fraction)

if alignment_fasta:
    aligned = parse_fasta_alignment(alignment_fasta)
    anchor_aligned = aligned["1FSZ_1"]
    all_aligned_seqs = list(aligned.values())
    n_seqs = len(all_aligned_seqs)

    CONSERVATION_THRESHOLD = 0.9  # >= 90% of aligned sequences share the same residue

    # Column 0 of the alignment corresponds to 1FSZ residue 1 (the FASTA/SEQRES
    # numbering matches the PDB's own residue numbering for this entry).
    residue_number = 1
    for col in range(len(anchor_aligned)):
        anchor_char = anchor_aligned[col]
        if anchor_char == "-":
            continue  # this column is a gap in 1FSZ itself -- no 1FSZ residue here
        matches = sum(1 for seq in all_aligned_seqs if seq[col] == anchor_char)
        fraction = matches / n_seqs
        if fraction >= CONSERVATION_THRESHOLD:
            conserved_residues.append((residue_number, anchor_char, fraction))
        residue_number += 1

    print(f"Alignment: {n_seqs} sequences, {len(anchor_aligned)} columns")
    print(f"Conserved positions (>= {CONSERVATION_THRESHOLD:.0%} identity): {len(conserved_residues)}")

    # Only positions inside 1FSZ's actually-modeled range (23-356) have 3D coordinates
    # to visualize -- filter to those for the structure view below.
    visualizable_conserved = [r for r in conserved_residues if 23 <= r[0] <= 356]
    print(f"Of those, {len(visualizable_conserved)} fall within 1FSZ's modeled range (23-356) and can be shown on the structure.")
else:
    visualizable_conserved = []
    print("No alignment available -- conservation analysis skipped.")


No alignment available -- conservation analysis skipped.


In [21]:
# Visualize 1FSZ with the conserved positions highlighted on the cartoon
view = py3Dmol.view(query="pdb:1FSZ", width=800, height=500)
view.setStyle({"cartoon": {"color": "lightgrey"}})

conserved_resi = [r[0] for r in visualizable_conserved]
if conserved_resi:
    view.addStyle(
        {"resi": conserved_resi},
        {"stick": {"color": "red"}},
    )
    view.addStyle(
        {"resi": conserved_resi},
        {"cartoon": {"color": "red"}},
    )

# Show the bound GDP too, for the interpretation question below
view.addStyle({"resn": "GDP"}, {"stick": {"colorscheme": "greenCarbon"}})

view.zoomTo()
view.show()


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

**Interpretation:** does conservation cluster near the bound GDP (shown in green
above), or is it spread evenly across the whole structure? In a live-verified run of this
exact pipeline, several of the conserved positions landed in the residue ranges known to
contact the bound GDP (roughly residues 45-135 and 160-215) — consistent with FtsZ/tubulin
biology: the nucleotide-binding site is the most functionally constrained part of a
GTPase, so it is under the strongest evolutionary pressure to stay identical across
species. **Your own run may find a different exact set of conserved positions** — which
specific ~10 sequences get selected depends on the live query results at the moment you
run this, not just on 1FSZ. Look at where *your* highlighted residues actually sit before
concluding this pattern holds; a single overlap is not proof, but a strong, evenly-spread
non-overlap would be worth taking seriously as a surprise, not brushing past.

---

## 📋 Character Sheet — choose your capstone protein

This is the **one** protein-selection decision you make in this whole course. Unlike
the worked example above (1FSZ), which is fixed and shared by the whole class, this
protein is yours: you'll carry it forward as your own parallel "your turn" track in
ex02 (AlphaFold), ex03 (molecular dynamics), and ex04 (docking), and it is the subject
of your final project.

Fill in the fields below (as a comment or markdown, whichever your instructor asks
for):

- **PDB ID:**
- **Protein name:**
- **Organism:**
- **Why this one interests you:**
- **One open question you already have about it:**


---

# Your Turn: Run This Analysis on Your Own Protein

This is the **only** graded exercise in this notebook. Everything above was a fully
worked walkthrough on 1FSZ — nothing there was left blank. Below, using the protein you
picked in the Character Sheet, repeat the same analysis, end to end. This consolidates
everything demonstrated above into one checklist; it is not organized as several separate
numbered exercises, because it is genuinely one connected analysis.

## Your Tasks

**1. Programmatic query, rooted in your own protein**
- Fetch your protein's own sequence (same pattern as the "fetch 1FSZ's own sequence"
  cell above).
- Run a sequence search rooted on it (same `find_best_structure`/sequence-search pattern
  above) to find related structures.
- Report: how many related structures did you find? Is your chosen structure itself the
  top hit?

**2. Structural-quality classification**
- Batch-process your hits the same way (`extract_structure_info`), and apply the same
  `classify_quality` tiers (or your own justified variant) to build a real table for your
  protein, not a fictional one.

**3. Critical evaluation (drug design vs. dynamics)**
- Using your own classification table: which structure would you pick for a drug-design
  question, and which for a flexibility question? Justify both choices explicitly,
  referencing actual resolution/R-free/method values from your table — model your answer
  on the worked "Worked critical evaluation" section above, don't just restate it.

**4. Visualization**
- Cartoon view colored by secondary structure.
- Cartoon view colored by domain or motif (from UniProt/PDB annotations or literature —
  cite where the domain boundaries came from).
- B-factor coloring (X-ray) or bundle visualization (NMR).
- If your structure has a ligand or cofactor: zoom in on it and the residues contacting
  it, the same way the GDP view above does for 1FSZ.

**5. B-factor analysis, with a real predict-first prediction**
- **Before running any plot**, write down your prediction: which region of your protein
  do you expect to be most flexible, and why? This is the actual graded predict-first
  artefact — the worked 1FSZ version above is a model of the reasoning, not something to
  copy the conclusion of.
- Then run the B-factor analysis and compare against your prediction. Were you right?
  What does the actual pattern tell you biologically?

**6. 🔭 Frontier: AlphaFold confidence guess**
- Following the same guardrail contract as the 1FSZ Frontier item above (⏭ Lecture 6 ·
  💡 use the B-factors/missing residues you just found · 🎁 bonus, does not block
  finishing · ✅ "I predict AlphaFold will be [confident/unconfident] about [region]
  because ___" is a complete answer): make your own prediction for your own protein.
  This is the one that's actually required of you — the 1FSZ version was a worked
  demonstration, not a substitute.

**7. Multiple sequence alignment and conserved-region visualization (required)**
- Repeat the MSA pipeline above: sequence-search hit list → select up to 10 sequences
  spread across the ranked list (skip this step with a clear note if you have fewer than
  3 related structures — that's a real, informative outcome, not a failure) → submit to
  EBI Clustal Omega → compute per-column conservation anchored on your own structure →
  visualize the conserved residues on your structure.
- **Interpretation:** do the conserved residues cluster somewhere meaningful (an active
  site, a ligand pocket, an interface) or are they spread out? State what you'd expect
  biologically *before* looking, the same predict-first habit as the B-factor task.
- ⚠️ Not every protein has as rich a set of PDB relatives as FtsZ (252 hits). If your
  search returns very few related structures, say so explicitly and explain what that
  itself might mean (a rarely-crystallized protein, a narrow taxonomic distribution,
  etc.) rather than forcing the analysis to look the same as the worked example.

## Your Work Area
